# nb12b — Cross-session matching audit (A ↔ C)

**Question:** why do only ~60% of the m=1 simple cells (found in Session C) carry through to
Session A for temporal extraction?

**What this separates:**
- **Biology/pipeline** — the Allen pipeline segments each session independently, then *associates*
  ROIs across sessions and assigns shared `cell_specimen_id`s. That association is partial, so not
  every Session C cell has a Session A counterpart. If the m=1 match rate ≈ the *all-cell* C→A match
  rate, the loss is just the pipeline yield — nothing is wrong.
- **Bug** — a wrong Session A experiment mapping, or m=1 `cell_id`s that aren't actually cross-session
  `cell_specimen_id`s. Caught by: (i) the mapped A experiment's `session_type` must be `three_session_A`,
  and (ii) every m=1 `cell_id` must be present in its **own** Session C cell list (sanity).

In [1]:
import os, numpy as np, pandas as pd
from pathlib import Path
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

base_dir    = Path(os.path.expanduser('~/dev/neuroscience/v1-dimensionality-study'))
cache_dir   = base_dir / 'data' / 'cache'
lists_dir   = base_dir / 'data' / 'experiment_lists'
outputs_dir = base_dir / 'outputs' / 'rf_params'
boc = BrainObservatoryCache(manifest_file=str(cache_dir / 'manifest.json'))

batch_df = pd.read_csv(lists_dir / 'all_l23_excitatory_experiments.csv')

# One experiment per container per session family. contains('A') hits only three_session_A;
# contains('C') hits three_session_C AND three_session_C2 (one per container) — that's the
# session the m=1 cells were analysed in.
exp_A = batch_df[batch_df['session_type'].str.contains('A')]
exp_C = batch_df[batch_df['session_type'].str.contains('C')]
container_to_exp_A = dict(zip(exp_A['experiment_container_id'], exp_A['id']))
container_to_exp_C = dict(zip(exp_C['experiment_container_id'], exp_C['id']))

m1 = pd.read_csv(outputs_dir / 'order_full_v2' / 'rf_gallery' / 'm1_neuron_dataset.csv')
m1['container_id'] = m1['container_id'].astype(int)
by_container = m1.groupby('container_id')['cell_id'].apply(lambda s: [int(x) for x in s]).to_dict()
print(f'{len(m1)} m=1 cells across {len(by_container)} containers')

31 m=1 cells across 13 containers


In [2]:
rows = []
for cid, m1_ids in by_container.items():
    a_exp = container_to_exp_A.get(cid)
    c_exp = container_to_exp_C.get(cid)
    if a_exp is None or c_exp is None:
        rows.append({'container': cid, 'note': 'missing A or C experiment mapping'}); continue

    dsA = boc.get_ophys_experiment_data(a_exp)
    dsC = boc.get_ophys_experiment_data(c_exp)

    # (i) mapping sanity — the A experiment must really be Session A
    a_type = batch_df.loc[batch_df['id'] == a_exp, 'session_type'].iloc[0]

    A_ids = set(int(x) for x in dsA.get_cell_specimen_ids())
    C_ids = set(int(x) for x in dsC.get_cell_specimen_ids())
    overlap = A_ids & C_ids

    # (ii) m=1 sanity + match
    m1_in_C = [c for c in m1_ids if c in C_ids]     # should equal m1_ids
    m1_in_A = [c for c in m1_ids if c in A_ids]

    rows.append({
        'container':      cid,
        'A_type':         a_type,
        'n_cells_A':      len(A_ids),
        'n_cells_C':      len(C_ids),
        'A∩C':            len(overlap),
        'allcell_C→A_%':  round(100*len(overlap)/max(len(C_ids),1), 1),
        'n_m1':           len(m1_ids),
        'm1_in_C':        len(m1_in_C),          # sanity: == n_m1 or IDs aren't cell_specimen_ids
        'm1_in_A':        len(m1_in_A),
        'm1_C→A_%':       round(100*len(m1_in_A)/max(len(m1_ids),1), 1),
    })

audit = pd.DataFrame(rows)
audit

,container,A_type,n_cells_A,n_cells_C,A∩C,allcell_C→A_%,n_m1,m1_in_C,m1_in_A,m1_C→A_%
0,511507650,three_session_A,205,164,97,59.1,1,1,1,100.0
1,511509529,three_session_A,215,189,146,77.2,2,2,1,50.0
2,511510670,three_session_A,292,208,150,72.1,2,2,1,50.0
3,511510718,three_session_A,226,202,136,67.3,2,2,2,100.0
4,511510855,three_session_A,266,284,199,70.1,5,5,2,40.0
5,650389885,three_session_A,207,181,134,74.0,2,2,1,50.0
6,652842570,three_session_A,244,190,145,76.3,3,3,2,66.7
7,653125128,three_session_A,224,211,155,73.5,5,5,3,60.0
8,661437138,three_session_A,86,102,61,59.8,1,1,0,0.0
9,661732156,three_session_A,107,189,76,40.2,2,2,1,50.0


In [3]:
# ---- Verdicts ----
bad_map = audit[audit['A_type'] != 'three_session_A']
bad_id  = audit[audit['m1_in_C'] != audit['n_m1']]

print('MAPPING CHECK  — A experiments not actually Session A :',
      'NONE ✓' if bad_map.empty else f'{len(bad_map)} ✗\n{bad_map}')
print('ID CHECK       — m=1 cells missing from their OWN Session C list :',
      'NONE ✓' if bad_id.empty else f'{len(bad_id)} ✗ (cell_id may not be cell_specimen_id)\n{bad_id}')

tot_m1     = audit['n_m1'].sum()
tot_m1_A   = audit['m1_in_A'].sum()
# cell-weighted all-cell C→A rate
w_allcell  = 100 * (audit['A∩C'].sum() / audit['n_cells_C'].sum())
w_m1       = 100 * (tot_m1_A / tot_m1)

print(f'\nm=1 matched into Session A : {tot_m1_A}/{tot_m1}  ({w_m1:.0f}%)')
print(f'ALL-cell C→A match rate    : {w_allcell:.0f}%  (cell-weighted across the 13 containers)')
print(f'\nInterpretation:')
print(f'  if m1 rate ≈ all-cell rate → loss is the Allen matching yield, not m=1-specific.')
print(f'  if m1 rate ≪ all-cell rate → m=1 cells are being lost preferentially (investigate).')

MAPPING CHECK  — A experiments not actually Session A : NONE ✓
ID CHECK       — m=1 cells missing from their OWN Session C list : NONE ✓

m=1 matched into Session A : 19/31  (61%)
ALL-cell C→A match rate    : 71%  (cell-weighted across the 13 containers)

Interpretation:
  if m1 rate ≈ all-cell rate → loss is the Allen matching yield, not m=1-specific.
  if m1 rate ≪ all-cell rate → m=1 cells are being lost preferentially (investigate).
